# Fraud Detection using Machine Learning

## Objective:
To build a model to detect fraudulent transactions and identify key factors influencing fraud.

In [1]:
# Install libraries
!pip install pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost

In [2]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [3]:
# Load dataset
df = pd.read_csv("fraud.csv")
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 1. Data Cleaning including missing values, outliers and multicollinearity

The dataset was cleaned by handling missing values and ensuring consistency in the target variable. Missing values were replaced with appropriate defaults to maintain data integrity.

Outliers were not removed aggressively because fraudulent transactions are often extreme in nature and may appear as outliers. Removing them would result in loss of important fraud-related data.

Multicollinearity was checked using correlation analysis to ensure that highly correlated features do not negatively impact model performance.

In [4]:
#data cleaning
df['isFraud'] = pd.to_numeric(df['isFraud'], errors='coerce')
df['isFraud'] = df['isFraud'].fillna(0).astype(int)


df = df.fillna(0)


df.drop(['nameOrig', 'nameDest'], axis=1, inplace=True)

In [5]:
# Feature Engineering
df['balance_diff'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['amount_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)

## 3. How did you select variables to be included in the model?

Feature selection was done based on domain understanding, correlation analysis, and feature importance. New features such as balance difference and transaction ratio were created to capture hidden patterns.

Irrelevant features such as transaction IDs were removed, as they do not contribute to prediction and can increase model complexity unnecessarily.

In [6]:
# Encode only 'type'
if 'type' in df.columns:
    df = pd.get_dummies(df, columns=['type'], drop_first=True)

In [7]:
# Define X and y
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## 2. Describe your fraud detection model in elaboration

The fraud detection model is built using LightGBM, a gradient boosting framework that constructs decision trees sequentially. Each tree improves the errors made by previous ones.

LightGBM was chosen due to its efficiency, scalability, and ability to handle large datasets. It also performs well with imbalanced data, making it suitable for fraud detection tasks.

In [8]:
# Take small sample for faster comparison
df_sample = df.sample(n=100000, random_state=42)

X_sample = df_sample.drop('isFraud', axis=1)
y_sample = df_sample['isFraud']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sample, y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

In [9]:
# Model Comparison
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

models = {
    "Random Forest": RandomForestClassifier(n_estimators=50),
    "XGBoost": XGBClassifier(n_estimators=50, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(class_weight='balanced')
}

results = {}

for name, model in models.items():
    print(f"\n{name} Performance:\n")

    model.fit(X_train_s, y_train_s)
    y_pred = model.predict(X_test_s)

    print(classification_report(y_test_s, y_pred))

    report = classification_report(y_test_s, y_pred, output_dict=True)

    results[name] = {
        "Precision": report.get('1', {}).get('precision', 0),
        "Recall": report.get('1', {}).get('recall', 0),
        "F1-score": report.get('1', {}).get('f1-score', 0)
    }


Random Forest Performance:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19972
           1       1.00      1.00      1.00        28

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000


XGBoost Performance:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19972
           1       0.96      0.79      0.86        28

    accuracy                           1.00     20000
   macro avg       0.98      0.89      0.93     20000
weighted avg       1.00      1.00      1.00     20000


LightGBM Performance:

[LightGBM] [Info] Number of positive: 113, number of negative: 79887
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002621 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can se

In [10]:
#Comparison table
results_df = pd.DataFrame(results).T
print("\nModel Comparison:\n")
print(results_df)


Model Comparison:

               Precision    Recall  F1-score
Random Forest   1.000000  1.000000  1.000000
XGBoost         0.956522  0.785714  0.862745
LightGBM        0.962963  0.928571  0.945455


## 4. Demonstrate the performance of the model using best set of tools

The performance of the models was evaluated using metrics such as Precision, Recall, F1-score, Accuracy, and ROC-AUC score.

Since the dataset is highly imbalanced, Recall was given more importance as it measures the ability of the model to correctly identify fraudulent transactions. Missing fraud cases can lead to significant financial loss, so improving recall is critical.

Model comparison was performed using a sampled subset of the data to reduce computation time, while the final model was trained on the full dataset. A confusion matrix and ROC curve were also used to visualize model performance and classification ability.

In [11]:
# FINAL MODEL USING FULL DATA

X = df.drop('isFraud', axis=1)
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Train final model
model = LGBMClassifier(class_weight='balanced')

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 6570, number of negative: 5083526
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.145754 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2048
[LightGBM] [Info] Number of data points in the train set: 5090096, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [12]:
#final predictions
y_prob = model.predict_proba(X_test)[:,1]

# Threshold tuning (important for fraud detection)
y_pred = (y_prob > 0.3).astype(int)

In [13]:
#Model evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC Score:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9998561913174132

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.90      0.99      0.95      1643

    accuracy                           1.00   1272524
   macro avg       0.95      1.00      0.97   1272524
weighted avg       1.00      1.00      1.00   1272524


Confusion Matrix:

[[1270707     174]
 [      9    1634]]

ROC-AUC Score: 0.9986569881227739


## 5. What are the key factors that predict fraudulent customers?

The model identified several key factors that contribute to fraudulent transactions. These include transaction amount, balance difference, and transaction-to-balance ratio.

Large transaction amounts and sudden drops in account balance are strong indicators of suspicious activity. Additionally, abnormal transaction patterns compared to the available balance also play a significant role in identifying fraud.

These features help capture unusual financial behavior, which is commonly associated with fraudulent activities.

## 6. Do these factors make sense? If yes, how?

Yes, these factors make logical sense and align with real-world fraud behavior.

Fraudulent transactions often involve large withdrawals or transfers, leading to sudden drops in account balance. Similarly, transactions that are unusually high compared to the available balance indicate suspicious activity.

These patterns reflect how fraudsters attempt to quickly extract money, making the identified factors meaningful and reliable.

## 7. What kind of prevention should be adopted while the company updates its infrastructure?

To prevent fraudulent transactions, companies should implement real-time fraud detection systems that continuously monitor transaction activities.

Additional preventive measures include multi-factor authentication, transaction limits, and anomaly detection systems powered by machine learning. Regular security audits and system updates should also be conducted to minimize vulnerabilities.

These strategies help in identifying suspicious activities early and reducing the risk of financial loss.

## 8. Assuming these actions have been implemented, how would you determine if they work?

The effectiveness of the implemented measures can be evaluated by comparing fraud detection rates before and after implementation.

Key performance indicators include reduction in fraudulent transactions, improvement in recall, and decrease in false positives. Monitoring customer feedback and system performance over time also helps in assessing effectiveness.

Continuous evaluation ensures that the system adapts to evolving fraud patterns and remains effective.